# 🏥 Medical ASR — Whisper LoRA vs DoRA vs Wav2Vec2
**Domain adaptation study — Medical Speech, Transcription & Intent dataset**

We take OpenAI Whisper-small (pre-trained on 680K hours of general speech) and adapt it
to medical speech using two parameter-efficient methods (LoRA and DoRA), then compare
both against a fully fine-tuned Wav2Vec2-base model using WER and CER metrics.

| Step | Description | Est. Time (P100) |
|------|-------------|------------------|
| 0 | GPU check | instant |
| 1 | Install dependencies | ~2 min |
| 2 | Load, clean & split dataset | ~2 min |
| 3 | Baseline evaluation (Whisper, no fine-tuning) | ~5 min |
| 4 | Feature extraction for Whisper models | ~30 min |
| 5 | Whisper + LoRA — train & evaluate | ~4 hrs |
| 6 | Whisper + DoRA — train & evaluate | ~4 hrs |
| 7 | Wav2Vec2 — train & evaluate | ~2 hrs |
| 8 | Final results comparison table | instant |
| 9 | Export & merge models | ~10 min |
| 10 | Inference demo | instant |

**Total estimated time: ~11 hours on Kaggle P100 (free quota: 30 hrs/week)**

> **Reusing a Colab checkpoint?** If you already trained LoRA in Colab and saved the
> adapter to Google Drive, see Cell 5b to download it and skip the 4-hour LoRA training.

---
## 0️⃣ GPU Check — Must Pass Before Continuing

In [2]:
# ── Verify GPU is available ───────────────────────────────────────────────────
# If this cell raises RuntimeError, go to:
#   Notebook Settings (top-right gear icon) → Accelerator → GPU P100
# Then re-run from the beginning.

!nvidia-smi
import torch

print('\n' + '='*55)
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {props.total_memory / 1e9:.1f} GB')
    print(f'PyTorch        : {torch.__version__}')
else:
    raise RuntimeError(
        '❌ No GPU found!\n'
        'Go to: Notebook Settings → Accelerator → GPU P100'
    )
print('='*55)

Sat Mar 28 10:59:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## 1️⃣ Install Dependencies

In [3]:
# ── Install all required libraries ───────────────────────────────────────────
# Pin transformers to 4.40.0 to avoid API changes that break LoRA training.
# All other packages use latest stable versions.
#
# Library roles:
#   transformers  — Whisper + Wav2Vec2 models, Trainer, Seq2SeqTrainer
#   datasets      — HuggingFace Dataset objects (memory-mapped Arrow format)
#   accelerate    — backend for distributed/fp16 training inside Trainer
#   peft          — LoRA / DoRA wrappers (get_peft_model, LoraConfig)
#   soundfile     — low-level .wav file I/O (used by librosa internally)
#   librosa       — audio loading + resampling to 16 kHz
#   jiwer         — WER / CER metric computation
#   scikit-learn  — train_test_split for 80/10/10 split

# Fix NumPy / SciPy / scikit-learn compatibility
!pip -q install --upgrade \
    transformers==4.46.3 \
    peft==0.13.2 \
    accelerate==0.34.2 \
    datasets==2.20.0 \
    soundfile==0.12.1 \
    librosa==0.10.2 \
    jiwer==3.0.4 \
    tqdm==4.66.4

# Do NOT pin numpy, scipy, scikit-learn, pandas — use whatever Kaggle has pre-installed
# Kaggle's base image already has compatible versions of all four

print('✅ Libraries installed.')

# Hard restart to flush any cached incompatible modules from memory
#import IPython
#IPython.Application.instance().kernel.do_shutdown(restart=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 93.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import os

# ── Clear every distributed env var that could trigger torch.distributed ──────
# These get set by Kaggle's runtime when 2 GPUs are present.
# If ANY of WORLD_SIZE / RANK / LOCAL_RANK exist, accelerate tries to call
# init_process_group — which then demands MASTER_ADDR and crashes.
# Solution: remove them all so accelerate sees a plain single-process run.
for var in [
    "WORLD_SIZE", "RANK", "LOCAL_RANK",
    "MASTER_ADDR", "MASTER_PORT",
    "TORCHELASTIC_RESTART_COUNT",
    "TORCHELASTIC_MAX_RESTARTS",
    "TORCHELASTIC_RUN_ID",
    "GROUP_RANK", "ROLE_RANK", "ROLE_NAME",
]:
    os.environ.pop(var, None)

# ── Tell accelerate explicitly: one process, one GPU, no distribution ─────────
os.environ["ACCELERATE_NUM_PROCESSES"] = "1"

# ── Write the accelerate default config ───────────────────────────────────────
import pathlib
cfg_path = pathlib.Path.home() / ".cache/huggingface/accelerate/default_config.yaml"
cfg_path.parent.mkdir(parents=True, exist_ok=True)
cfg_path.write_text("""\
compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
rdzv_backend: static
same_network: true
use_cpu: false
""")

# ── Patch PartialState before TrainingArguments is ever imported ───────────────
# This is the nuclear option: monkey-patch accelerate's PartialState.__init__
# to never call init_process_group regardless of what it detects.
from accelerate.state import PartialState
import accelerate.state as _acc_state
from accelerate.utils import DistributedType

_original_partial_state_init = PartialState.__init__

def _patched_partial_state_init_DISABLED(self, cpu=False, **kwargs):
    # Skip distributed setup entirely — force single-process state
    self.__dict__.clear()
    self._shared_state = {}   
    object.__setattr__(self, '_shared_state', PartialState.__dict__.get('_shared_state', {}))
    # Set the minimum attributes TrainingArguments reads
    self.process_index        = 0
    self.local_process_index  = 0
    self.num_processes        = 1
    self.distributed_type     = DistributedType.NO
    self.device               = __import__('torch').device('cuda:0')
    # self.is_last_process      = True  # disabled: avoid setting read-only accelerate properties
    self.use_distributed      = False

# PartialState.__init__ = _patched_partial_state_init_DISABLED  # disabled (prevents corrupting accelerate state)
print("✅ Skipping PartialState monkey-patching (safe single-process config)")
print("✅ All distributed env vars cleared")
print("✅ Accelerate config written")
print("\n🔒 From this point on: single GPU (cuda:0), no DataParallel, no torch.distributed")

✅ Skipping PartialState monkey-patching (safe single-process config)
✅ All distributed env vars cleared
✅ Accelerate config written

🔒 From this point on: single GPU (cuda:0), no DataParallel, no torch.distributed


---
## 2️⃣ Configure Paths, Load & Clean Dataset

**Before running this cell:**
1. Click **+ Add Data** (top-right of the Kaggle editor)
2. Search: `medical speech transcription intent`
3. Click **Add** — it will appear at `/kaggle/input/medical-speech-transcription-and-intent/`

**Why we re-split with 80/10/10 instead of using the original folders:**
The original dataset puts ~89% of files in the `test` folder and only ~6% in `train`.
That is backwards for training. We ignore the original split completely and
re-split all clean data ourselves: 80% train / 10% val / 10% test.

In [5]:
import os

# ── Kaggle dataset base path ──────────────────────────────────────────────────
# Kaggle may place the dataset in slightly different subdirectory structures
# depending on how it was uploaded. We try both known layouts and pick whichever exists.
POSSIBLE_BASES = [
    '/kaggle/input/medical-speech-transcription-and-intent/medical speech transcription and intent/Medical Speech, Transcription, and Intent',
    '/kaggle/input/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent',
    '/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent'
]
KAGGLE_BASE = next((b for b in POSSIBLE_BASES if os.path.exists(b)), None)

if KAGGLE_BASE is None:
    # Print what Kaggle actually mounted so the user can debug the path
    print('❌ Dataset not found. Available input directories:')
    for root, dirs, _ in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        if depth < 4:
            print(f'  {root}')
    raise FileNotFoundError(
        'Dataset not found. Add it via + Add Data → '
        'search "medical speech transcription intent" → Add'
    )

# ── Input paths (read-only on Kaggle) ────────────────────────────────────────
CSV_PATH  = os.path.join(KAGGLE_BASE, 'overview-of-recordings.csv')
TRAIN_DIR = os.path.join(KAGGLE_BASE, 'recordings', 'train')
VAL_DIR   = os.path.join(KAGGLE_BASE, 'recordings', 'validate')
TEST_DIR  = os.path.join(KAGGLE_BASE, 'recordings', 'test')

# ── Output paths (writable — /kaggle/working) ─────────────────────────────────
# Everything saved here is downloadable from the Kaggle output panel after the run.
WORK_DIR         = '/kaggle/working'
OUTPUT_DIR_LORA  = f'{WORK_DIR}/whisper-lora-training'   # training checkpoints
OUTPUT_DIR_DORA  = f'{WORK_DIR}/whisper-dora-training'
OUTPUT_DIR_W2V   = f'{WORK_DIR}/wav2vec2-training'
ADAPTER_DIR_LORA = f'{WORK_DIR}/whisper-lora-adapter'    # lightweight LoRA weights only
ADAPTER_DIR_DORA = f'{WORK_DIR}/whisper-dora-adapter'
W2V_SAVE_DIR     = f'{WORK_DIR}/wav2vec2-saved'          # full Wav2Vec2 model
MERGED_DIR_LORA  = f'{WORK_DIR}/whisper-lora-merged'     # LoRA baked into Whisper (standalone)
MERGED_DIR_DORA  = f'{WORK_DIR}/whisper-dora-merged'

# ── Sanity check — confirm all input paths exist ──────────────────────────────
print('📁 Path verification:')
print('-' * 65)
all_ok = True
for name, p in [('CSV', CSV_PATH), ('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    exists = os.path.exists(p)
    count  = len(os.listdir(p)) if exists and os.path.isdir(p) else '(file)'
    icon   = '✅' if exists else '❌'
    print(f'  {icon} {name:6s} → items={str(count):>6}  path={p}')
    if not exists:
        all_ok = False
if not all_ok:
    raise FileNotFoundError('One or more paths missing — check the dataset was added correctly.')
print(f'\n  Base : {KAGGLE_BASE}')
print('\n✅ All paths verified.')

📁 Path verification:
-----------------------------------------------------------------
  ✅ CSV    → items=(file)  path=/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent/overview-of-recordings.csv
  ✅ train  → items=   381  path=/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent/recordings/train
  ✅ val    → items=   385  path=/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent/recordings/validate
  ✅ test   → items=  5895  path=/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent/recordings/test

  Base : /kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent

✅ All paths verified.


In [6]:
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

# ── Step 1: Load the CSV ──────────────────────────────────────────────────────
# The CSV is the master metadata file with one row per recording.
# It contains: file_name, phrase (ground-truth transcript), audio quality scores.
df = pd.read_csv(CSV_PATH)
# Normalise transcript: lowercase + strip whitespace.
# This ensures WER/CER comparisons are case-insensitive and whitespace-clean.
df['text'] = df['phrase'].astype(str).str.strip().str.lower()
print(f'📄 CSV loaded: {len(df):,} rows, {len(df.columns)} columns')

# ── Step 2: Resolve audio paths ───────────────────────────────────────────────
# The CSV contains only the bare filename (e.g. 1249120_43453425_58166571.wav).
# We search all three original folders and build a filename → full_path index.
# This works regardless of which folder a file was originally placed in.
file_index = {}
for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if os.path.isdir(d):
        for fname in os.listdir(d):
            file_index[fname] = os.path.join(d, fname)

df['audio_path'] = df['file_name'].map(file_index)
found = df['audio_path'].notna().sum()
print(f'🔍 Audio files matched: {found:,} / {len(df):,}')

# ── Step 3: Quality filter ────────────────────────────────────────────────────
# We keep only clean recordings to avoid training the model on bad audio.
# Thresholds chosen based on dataset documentation:
#   overall_quality >= 3.33  → keeps approx top 75% by quality score
#   no heavy clipping        → heavy clipping distorts the waveform badly
#   no heavy noise           → background noise confuses the model
#   non-empty transcript     → empty labels would corrupt the loss function
clean = df.dropna(subset=['audio_path']).copy()
before = len(clean)
clean = clean[clean['overall_quality_of_the_audio'] >= 3.33]
clean = clean[clean['audio_clipping'].isin(['no_clipping', 'light_clipping'])]
clean = clean[clean['background_noise_audible'].isin(['no_noise', 'light_noise'])]
clean = clean[clean['text'].str.len() > 0]
after = len(clean)
print(f'🧹 Quality filter: {before:,} → {after:,} kept  ({before - after:,} removed)')

# ── Step 4: 80 / 10 / 10 split ───────────────────────────────────────────────
# We IGNORE the original train/validate/test folder split because the original
# distribution is extremely unbalanced: ~89% test, ~6% train, ~6% validate.
# Training on only 6% of the data with 89% held out for test is useless.
# random_state=42 makes the split reproducible — running again gives same split.
train_df, temp_df = train_test_split(clean, test_size=0.20, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

# Keep only the columns we need
train_df = train_df[['audio_path', 'text']].reset_index(drop=True)
val_df   = val_df[['audio_path',   'text']].reset_index(drop=True)
test_df  = test_df[['audio_path',  'text']].reset_index(drop=True)

print(f'\n✂️  Final 80/10/10 split:')
print(f'   train : {len(train_df):,} samples')
print(f'   val   : {len(val_df):,} samples')
print(f'   test  : {len(test_df):,} samples')
print(f'\n📝 Sample transcripts from training set:')
for _, row in train_df.head(3).iterrows():
    print(f"   → {row['text'][:80]}")

📄 CSV loaded: 6,661 rows, 14 columns
🔍 Audio files matched: 6,661 / 6,661
🧹 Quality filter: 6,661 → 6,106 kept  (555 removed)

✂️  Final 80/10/10 split:
   train : 4,884 samples
   val   : 611 samples
   test  : 611 samples

📝 Sample transcripts from training set:
   → i often get a sharp pain in my chest and i can't tell what i'm doing that might 
   → i can't carry anything i have a pain in my shoulder
   → i feel something hurt me in taking breath and i cant take my breath


---
## 3️⃣ Baseline Evaluation — Whisper-small (No Fine-Tuning)

Before we train anything, we measure how well Whisper-small performs on medical speech
**out of the box**. This gives us the baseline scores to beat.

**Why use a fixed 200-sample subset?**
Running inference on the full test set (~600 samples) for 4 different models would take
hours. We fix a random 200-sample subset with `random_state=42` and reuse it for ALL
model comparisons — this keeps the comparison perfectly fair.

In [7]:
import torch
import librosa
from tqdm import tqdm
from jiwer import wer, cer
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers.generation.configuration_utils import GenerationConfig

# ── Global constants used throughout the notebook ────────────────────────────
MODEL_NAME = 'openai/whisper-small'  # 244M params, good balance of speed/accuracy
TARGET_SR  = 16_000                  # Whisper requires 16 kHz audio
N_EVAL     = 200                     # samples used for all model evaluations
device     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {device}')

# ── Load Whisper processor and baseline model ─────────────────────────────────
# The processor contains:
#   - feature_extractor: converts raw audio → log-mel spectrogram (80 mel bins)
#   - tokenizer: converts text ↔ token IDs for the decoder
print(f'⬇️  Loading {MODEL_NAME}...')
processor      = WhisperProcessor.from_pretrained(MODEL_NAME)
baseline_model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# ── Fix generation config (required for transformers >= 4.38) ─────────────────
# Older code set generation parameters on model.config, but newer transformers
# moved them to model.generation_config and raises an error if model.config
# still has them. We delete them from config and set them properly on
# generation_config instead. This pattern is applied to every model we load.
if hasattr(baseline_model.config, 'suppress_tokens'):    del baseline_model.config.suppress_tokens
if hasattr(baseline_model.config, 'forced_decoder_ids'): del baseline_model.config.forced_decoder_ids
baseline_model.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
baseline_model.generation_config.forced_decoder_ids = None  # allow any language
baseline_model.generation_config.suppress_tokens    = []    # don't suppress any tokens
baseline_model.eval().to(device)
print('✅ Baseline model loaded.')

# ── Transcription function shared by all Whisper models ───────────────────────
# Steps:
#   1. librosa.load — decode .wav and resample to TARGET_SR (16 kHz)
#   2. processor    — convert waveform to log-mel spectrogram tensor
#   3. model.generate — autoregressive decoding (beam search by default)
#   4. batch_decode — convert token IDs back to text string
#
# max_new_tokens=128 caps output length (medical phrases are short, 128 is more than enough).
# max_length=None prevents a conflict warning between max_new_tokens and max_length.
# language='english' forces English output, avoiding accidental translation.
# task='transcribe' prevents the model switching to translation mode.
def transcribe_whisper(model, audio_path):
    audio, _ = librosa.load(audio_path, sr=TARGET_SR)
    inputs   = processor(audio, sampling_rate=TARGET_SR, return_tensors='pt').to(device)
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_new_tokens=128,
            max_length=None,
            language='english',
            task='transcribe'
        )
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip().lower()

# ── Fixed evaluation subset ────────────────────────────────────────────────────
# IMPORTANT: same random_state=42 is used here and in all subsequent evals.
# This guarantees we always evaluate all models on the exact same 200 samples.
# Changing this would make comparisons between models unfair.
eval_subset = test_df.sample(N_EVAL, random_state=42).reset_index(drop=True)
refs        = eval_subset['text'].tolist()  # ground-truth transcripts (lowercase)

print(f'\n🔬 Evaluating baseline on {N_EVAL} fixed samples...')
baseline_preds = [
    transcribe_whisper(baseline_model, p)
    for p in tqdm(eval_subset['audio_path'].tolist(), desc='Baseline eval')
]

# ── Compute WER and CER ───────────────────────────────────────────────────────
# WER (Word Error Rate): fraction of words wrong
#   WER = (Substitutions + Deletions + Insertions) / Total_Reference_Words
#   WER = 0.0 means perfect transcription, 1.0 means completely wrong.
# CER (Character Error Rate): same but at character level.
#   CER is always lower than WER because getting one character wrong
#   ruins a word (WER) but is a small fraction of characters (CER).
baseline_wer = wer(refs, baseline_preds)
baseline_cer = cer(refs, baseline_preds)
print(f'\n📊 BASELINE — Whisper-small (no fine-tuning)')
print(f'   WER : {baseline_wer*100:.2f}%  ← target to beat')
print(f'   CER : {baseline_cer*100:.2f}%')

2026-03-28 11:00:25.522714: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774695625.741064      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774695625.800596      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774695626.296661      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774695626.296703      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774695626.296705      55 computation_placer.cc:177] computation placer alr

🖥️  Device: cuda
⬇️  Loading openai/whisper-small...


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

✅ Baseline model loaded.

🔬 Evaluating baseline on 200 fixed samples...


Baseline eval: 100%|██████████| 200/200 [01:12<00:00,  2.75it/s]


📊 BASELINE — Whisper-small (no fine-tuning)
   WER : 16.27%  ← target to beat
   CER : 7.11%


In [8]:
# ── Free baseline model from GPU memory ──────────────────────────────────────
# We delete the baseline model immediately after evaluation because:
#   - P100 has 16 GB VRAM; Whisper-small is ~1 GB loaded
#   - Keeping it loaded while training LoRA would waste memory and risk OOM
#   - We already have the scores stored in baseline_wer / baseline_cer
import gc
del baseline_model
torch.cuda.empty_cache()
gc.collect()
print('✅ Baseline model freed from GPU memory.')

✅ Baseline model freed from GPU memory.


---
## 4️⃣ Feature Extraction — ~30 min

We pre-process all audio files into model-ready tensors and store them on disk
using HuggingFace Datasets' Arrow format.

**Why pre-process instead of loading on-the-fly during training?**
Loading and processing audio during training creates a CPU bottleneck that starves
the GPU. Pre-processing once and storing to disk makes each training step much faster.

**What `prepare_batch` does:**
- `librosa.load` → reads the .wav file and resamples it to 16 kHz
- `feature_extractor` → converts the 1D waveform to an 80-bin log-mel spectrogram
  of shape (80, 3000) — this is the actual input Whisper's encoder sees
- `tokenizer` → converts the text transcript into a sequence of integer token IDs
  that the decoder is trained to predict

**Note:** This cell produces datasets for Whisper only. Wav2Vec2 uses raw waveforms
(not mel spectrograms), so it has its own prepare step in Cell 7.

In [9]:
import gc
import os
import time
import numpy as np
import librosa
import torch
from datasets import Dataset, concatenate_datasets
from dataclasses import dataclass
from typing import Any, Dict, List

# ── Chunked feature extraction — avoids both OOM and Dataset.map hang ─────────
# Strategy:
#   1. Process audio in chunks of CHUNK_SIZE examples
#   2. After each chunk, save to disk immediately and clear RAM
#   3. At the end, reload all chunks from disk and concatenate
#   This keeps peak RAM at ~CHUNK_SIZE × 80×3000 × 4 bytes ≈ 200 MB per chunk

CHUNK_SIZE = 50   # Process 50 files at a time — safe for 15 GB VRAM

def build_whisper_dataset_chunked(df, split_name, save_base, chunk_size=CHUNK_SIZE):
    total = len(df)
    n_chunks = (total + chunk_size - 1) // chunk_size
    chunk_dirs = []

    print(f"\n🔄 {split_name}: {total:,} samples in {n_chunks} chunks of {chunk_size}...")
    t0 = time.time()

    for chunk_idx in range(n_chunks):
        start = chunk_idx * chunk_size
        end   = min(start + chunk_size, total)
        chunk_df = df.iloc[start:end]

        rows = []
        for row in chunk_df.itertuples(index=False):
            try:
                audio, _ = librosa.load(row.audio_path, sr=TARGET_SR)
                feats = processor.feature_extractor(
                    audio, sampling_rate=TARGET_SR
                ).input_features[0]
                labels = processor.tokenizer(row.text).input_ids
                rows.append({
                    "input_features": np.asarray(feats, dtype=np.float32),
                    "labels": labels,
                })
            except Exception as e:
                print(f"   ⚠️  Skipping {row.audio_path}: {e}")
                continue

        # Save this chunk to disk immediately, then free RAM
        chunk_dir = f"{save_base}_chunk_{chunk_idx:04d}"
        Dataset.from_list(rows).save_to_disk(chunk_dir)
        chunk_dirs.append(chunk_dir)
        del rows
        gc.collect()

        # Progress
        elapsed = time.time() - t0
        rate    = (end) / elapsed if elapsed > 0 else 0
        eta     = (total - end) / rate if rate > 0 else 0
        print(f"   chunk {chunk_idx+1:3d}/{n_chunks} | {end:,}/{total:,} "
              f"| {rate:.1f} ex/s | ETA {eta/60:.1f} min")

    # Reload all chunks and concatenate into one Dataset
    print(f"   Concatenating {len(chunk_dirs)} chunks...")
    from datasets import load_from_disk
    all_chunks = [load_from_disk(d) for d in chunk_dirs]
    full_ds = concatenate_datasets(all_chunks)

    # Save final merged dataset and clean up chunk dirs
    full_ds.save_to_disk(save_base)
    import shutil
    for d in chunk_dirs:
        shutil.rmtree(d, ignore_errors=True)

    print(f"✅ {split_name} done: {len(full_ds):,} samples  → {save_base}")
    return full_ds


# ── Paths for caching to disk (Kaggle /kaggle/working/ survives session) ──────
TRAIN_HF_DIR = "/kaggle/working/whisper_train_hf"
VAL_HF_DIR   = "/kaggle/working/whisper_val_hf"
TEST_HF_DIR  = "/kaggle/working/whisper_test_hf"

# ── Check if cached datasets already exist (avoids re-processing on re-run) ───
from datasets import load_from_disk

def load_or_build(df, split_name, save_dir):
    if os.path.exists(save_dir):
        print(f"⚡ {split_name}: loading from cache at {save_dir}")
        return load_from_disk(save_dir)
    return build_whisper_dataset_chunked(df, split_name, save_dir)

train_hf = load_or_build(train_df, "train", TRAIN_HF_DIR)
val_hf   = load_or_build(val_df,   "val",   VAL_HF_DIR)
test_hf  = load_or_build(test_df,  "test",  TEST_HF_DIR)

print(f"\n✅ Feature extraction complete:")
print(f"   train : {len(train_hf):,} samples")
print(f"   val   : {len(val_hf):,} samples")
print(f"   test  : {len(test_hf):,} samples")

# ── Data Collator ─────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
gc.collect()
torch.cuda.empty_cache()
print("\n✅ Data collator ready.")


🔄 train: 4,884 samples in 98 chunks of 50...


Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

   chunk   1/98 | 50/4,884 | 16.0 ex/s | ETA 5.0 min


Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

   chunk   2/98 | 100/4,884 | 16.9 ex/s | ETA 4.7 min


...

   chunk  12/13 | 600/611 | 18.9 ex/s | ETA 0.0 min


Saving the dataset (0/1 shards):   0%|          | 0/11 [00:00<?, ? examples/s]

   chunk  13/13 | 611/611 | 18.7 ex/s | ETA 0.0 min
   Concatenating 13 chunks...


Saving the dataset (0/2 shards):   0%|          | 0/611 [00:00<?, ? examples/s]

✅ test done: 611 samples  → /kaggle/working/whisper_test_hf

✅ Feature extraction complete:
   train : 4,884 samples
   val   : 611 samples
   test  : 611 samples

✅ Data collator ready.


---
## 5️⃣ Model A — Whisper-small + LoRA (~4 hrs)

**What is LoRA?**
Instead of updating all 244M Whisper parameters, LoRA *freezes* the original weights
and injects small trainable matrices into the attention layers.
Each injected matrix is factored as two small matrices (rank r=16),
so the number of new parameters is tiny (~2.3M = ~1% of total).

Mathematically, if W is the original frozen weight, LoRA adds:
  `W_new = W + (B × A) × (alpha/r)`
where A and B are the small trainable matrices, alpha is a scaling factor.

**Why only q_proj and v_proj?**
These are the Query and Value projection matrices in each attention head.
Research shows that adapting Q and V gives the best accuracy/parameter tradeoff
for Whisper. Adding k_proj or out_proj increases parameters with diminishing returns.

**Cell 5b below:** If you already ran LoRA training in Google Colab and saved the adapter,
you can download it from Drive and skip the 4-hour training. Otherwise run 5a.

In [20]:
# =========================
# FINAL 1-CELL WHISPER LoRA TRAINING (KAGGLE SAFE)
# includes Accelerate reset fix before Seq2SeqTrainer(...)
# =========================

import os
import gc
import pathlib
import torch
from dataclasses import dataclass
from typing import Any, Dict, List

from transformers import (
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.generation.configuration_utils import GenerationConfig
from peft import LoraConfig, get_peft_model, TaskType

# ------------------------------------------------------------
# 0) Force Accelerate to single GPU / no distributed
# ------------------------------------------------------------
os.environ.pop("MASTER_ADDR", None)
os.environ.pop("MASTER_PORT", None)
os.environ.pop("WORLD_SIZE", None)
os.environ.pop("RANK", None)
os.environ.pop("LOCAL_RANK", None)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

cfg = pathlib.Path.home() / ".cache/huggingface/accelerate/default_config.yaml"
cfg.parent.mkdir(parents=True, exist_ok=True)
cfg.write_text("""\
compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
use_cpu: false
""")

# Fresh PartialState check
from accelerate.state import PartialState
from accelerate.utils import DistributedType

ps = PartialState()
#if ps.num_processes != 1 or ps.distributed_type != DistributedType.NO:
    # ps._shared_state["num_processes"] = 1  # disabled: do not mutate accelerate internals
    # ps._shared_state["distributed_type"] = DistributedType.NO  # disabled: do not mutate accelerate internals

print(f"num_processes   : {ps.num_processes}")
print(f"distributed_type: {ps.distributed_type}")
print(f"device          : {ps.device}")

# ------------------------------------------------------------
# 1) Cleanup
# ------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"training device : {device}")

# Keep only the columns Whisper needs
KEEP = {"input_features", "labels"}

def clean_ds(ds):
    drop_cols = [c for c in ds.column_names if c not in KEEP]
    if drop_cols:
        ds = ds.remove_columns(drop_cols)
    # ds.set_format(type="torch")  # disabled: NumPy 2.0 + datasets torch formatter crash
    ds.reset_format()  # keep python/numpy format; collator/processor will create torch tensors
    return ds

train_clean = clean_ds(train_hf)
val_clean   = clean_ds(val_hf)

print("train columns:", train_clean.column_names)
print("val columns  :", val_clean.column_names)
print("sample keys  :", train_clean[0].keys())

# ------------------------------------------------------------
# 2) Whisper collator: returns ONLY input_features + labels
# ------------------------------------------------------------
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        bos_id = self.processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos_id is not None:
            if (labels[:, 0] == bos_id).all():
                labels = labels[:, 1:]

        batch["labels"] = labels

        # Safety: Whisper must not receive input_ids
        batch.pop("input_ids", None)
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# sanity check
test_batch = data_collator([train_clean[0], train_clean[1]])
print("collator keys:", test_batch.keys())
for k, v in test_batch.items():
    print(f"{k}: {tuple(v.shape)}")

# ------------------------------------------------------------
# 3) Load fresh Whisper base model
# ------------------------------------------------------------
print(f"\n⬇️ Loading {MODEL_NAME} for LoRA...")
lora_base = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

if hasattr(lora_base.config, "suppress_tokens"):
    del lora_base.config.suppress_tokens
if hasattr(lora_base.config, "forced_decoder_ids"):
    del lora_base.config.forced_decoder_ids

lora_base.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
lora_base.generation_config.forced_decoder_ids = None
lora_base.generation_config.suppress_tokens = []

if hasattr(lora_base.config, "use_cache"):
    lora_base.config.use_cache = False

# ------------------------------------------------------------
# 4) Apply LoRA
# ------------------------------------------------------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    use_dora=False,
)

lora_model = get_peft_model(lora_base, lora_config)
lora_model.to(device)
lora_model.print_trainable_parameters()

# Use the underlying Whisper model for forward.
# PEFT's task-specific wrapper may pass input_ids even when we only provide input_features.
train_model = lora_model.get_base_model()
train_model.to(device)

# ------------------------------------------------------------
# 5) Training arguments
# ------------------------------------------------------------
lora_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR_LORA,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    warmup_steps=50,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    predict_with_generate=False,
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
)

# ------------------------------------------------------------
# 6) HARD RESET Accelerate state RIGHT BEFORE Trainer
# fixes: AcceleratorState object has no attribute distributed_type
# ------------------------------------------------------------
import accelerate.state as accel_state

# accel_state.PartialState._shared_state.clear()  # disabled: avoid corrupting accelerate state
# accel_state.AcceleratorState._shared_state.clear()  # disabled: avoid corrupting accelerate state

from accelerate.state import PartialState
ps = PartialState()
print("\nAfter reset:")
print("num_processes   :", ps.num_processes)
print("distributed_type:", ps.distributed_type)
print("device          :", ps.device)

gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 7) Trainer
# ------------------------------------------------------------
lora_trainer = Seq2SeqTrainer(
    model=train_model,
    args=lora_args,
    train_dataset=train_clean,
    eval_dataset=val_clean,
    data_collator=data_collator,
)

# ------------------------------------------------------------
# 8) Sanity forward pass
# ------------------------------------------------------------
print("\n🧪 Running sanity forward pass...")
sanity_batch = data_collator([train_clean[0], train_clean[1]])
sanity_batch = {k: v.to(device) for k, v in sanity_batch.items() if k in ("input_features", "labels")}

with torch.no_grad():
    sanity_out = train_model(**sanity_batch)

print("✅ Sanity forward pass OK")
print("sanity loss:", float(sanity_out.loss))

# ------------------------------------------------------------
# 9) Train
# ------------------------------------------------------------
print("\n🚀 Training LoRA...")
train_result = lora_trainer.train()

# ------------------------------------------------------------
# 10) Save adapter
# ------------------------------------------------------------
os.makedirs(ADAPTER_DIR_LORA, exist_ok=True)
lora_model.save_pretrained(ADAPTER_DIR_LORA)
processor.save_pretrained(ADAPTER_DIR_LORA)

print(f"\n✅ LoRA adapter saved -> {ADAPTER_DIR_LORA}")

# ------------------------------------------------------------
# 11) Cleanup
# ------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()

num_processes   : 1
distributed_type: DistributedType.NO
device          : cuda
training device : cuda:0
train columns: ['input_features', 'labels']
val columns  : ['input_features', 'labels']
sample keys  : dict_keys(['input_features', 'labels'])
collator keys: dict_keys(['input_features', 'labels'])
input_features: (2, 80, 3000)
labels: (2, 27)

⬇️ Loading openai/whisper-small for LoRA...
trainable params: 1,769,472 || all params: 243,504,384 || trainable%: 0.7267

After reset:
num_processes   : 1
distributed_type: DistributedType.NO
device          : cuda

🧪 Running sanity forward pass...
✅ Sanity forward pass OK
sanity loss: 4.814150810241699

🚀 Training LoRA...


Epoch,Training Loss,Validation Loss
0,0.441700,0.413438
1,0.270400,0.297241
2,0.211200,0.235808
4,0.053000,0.150268
5,0.026900,0.145030
6,0.026100,0.142248
8,0.018600,0.146969
9,0.010000,0.147889


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].



✅ LoRA adapter saved -> /kaggle/working/whisper-lora-adapter


In [21]:
# ── 5b: OPTIONAL — Load LoRA checkpoint from Google Drive ─────────────────────
# Use this INSTEAD OF Cell 5a if you already trained LoRA in Colab.
#
# How to get your Drive folder ID:
#   1. Open Google Drive → navigate to your whisper-lora-adapter folder
#   2. Right-click → Share → Copy link
#   3. The link contains: .../folders/XXXXXXXXXXXXXXXX
#   4. That XXXXXXXXXXXXXXXX is your DRIVE_FOLDER_ID
#
# DRIVE_FOLDER_ID = 'paste_your_folder_id_here'
#
# !pip -q install gdown
# import gdown
# os.makedirs(ADAPTER_DIR_LORA, exist_ok=True)
# gdown.download_folder(
#     f'https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}',
#     output=ADAPTER_DIR_LORA, quiet=False
# )
# from transformers import WhisperForConditionalGeneration
# from transformers.generation.configuration_utils import GenerationConfig
# from peft import PeftModel
# lora_base  = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
# lora_model = PeftModel.from_pretrained(lora_base, ADAPTER_DIR_LORA)
# lora_model.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
# lora_model.generation_config.forced_decoder_ids = None
# lora_model.generation_config.suppress_tokens    = []
# lora_model.eval().to(device)
# print('✅ LoRA loaded from Drive — skipping 5a training!')

print('ℹ️  Cell 5b is commented out. Fill in DRIVE_FOLDER_ID and uncomment to use a Drive checkpoint.')

ℹ️  Cell 5b is commented out. Fill in DRIVE_FOLDER_ID and uncomment to use a Drive checkpoint.


In [22]:
# ── 5c: Evaluate LoRA model ────────────────────────────────────────────────────
# We re-apply the generation config fix here because get_peft_model creates a
# new wrapper object that doesn't automatically inherit the fix from lora_base.
from jiwer import wer, cer
from transformers.generation.configuration_utils import GenerationConfig

# Clear any stale suppress_tokens from model.config before evaluating
if hasattr(lora_model.config, 'suppress_tokens'):    del lora_model.config.suppress_tokens
if hasattr(lora_model.config, 'forced_decoder_ids'): del lora_model.config.forced_decoder_ids
lora_model.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
lora_model.generation_config.forced_decoder_ids = None
lora_model.generation_config.suppress_tokens    = []
lora_model.eval()

# Evaluate on the same 200 samples used for the baseline (eval_subset, refs defined in Cell 3)
print(f'🔬 Evaluating LoRA on {N_EVAL} samples (same fixed subset as baseline)...')
lora_preds = [
    transcribe_whisper(lora_model, p)
    for p in tqdm(eval_subset['audio_path'].tolist(), desc='LoRA eval')
]

lora_wer = wer(refs, lora_preds)
lora_cer = cer(refs, lora_preds)
lora_improvement = (baseline_wer - lora_wer) / baseline_wer * 100
print(f'\n📊 MODEL A — Whisper-small + LoRA')
print(f'   WER : {lora_wer*100:.2f}%  (baseline was {baseline_wer*100:.2f}%)  → {lora_improvement:+.1f}% vs baseline')
print(f'   CER : {lora_cer*100:.2f}%')

🔬 Evaluating LoRA on 200 samples (same fixed subset as baseline)...


LoRA eval: 100%|██████████| 200/200 [01:13<00:00,  2.72it/s]


📊 MODEL A — Whisper-small + LoRA
   WER : 7.30%  (baseline was 16.27%)  → +55.2% vs baseline
   CER : 4.44%


In [23]:
# ── Free LoRA model from GPU before loading DoRA ─────────────────────────────
# Always free the previous model before loading the next one to avoid OOM.
import gc
del lora_model
torch.cuda.empty_cache()
gc.collect()
print('✅ LoRA model freed from GPU memory.')

✅ LoRA model freed from GPU memory.


---
## 6️⃣ Model B — Whisper-small + DoRA (~4 hrs)

**What is DoRA and how does it differ from LoRA?**

Standard LoRA adds a low-rank update: `W_new = W + B×A`

DoRA (Weight-Decomposed Low-Rank Adaptation) first decomposes the weight matrix
into two components: magnitude (||W||) and direction (W / ||W||).
It then applies LoRA only to the *direction* component while learning
the *magnitude* separately.

In practice this means DoRA is better at mimicking full fine-tuning behaviour
while keeping the same low parameter count as LoRA.
Published results show DoRA typically achieves 1–3% lower WER than standard LoRA.

The only code change from LoRA is: `use_dora=True` in the LoraConfig.

In [10]:
import gc, torch, os

# Delete everything from LoRA training
try: del lora_trainer
except: pass
try: del lora_model
except: pass
try: del lora_base
except: pass
try: del train_model
except: pass
try: del sanity_out
except: pass
try: del sanity_batch
except: pass
try: del test_batch
except: pass

gc.collect()
torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()

# Check VRAM
if torch.cuda.is_available():
    free  = torch.cuda.mem_get_info()[0] / 1e9
    total = torch.cuda.mem_get_info()[1] / 1e9
    print(f"VRAM free : {free:.1f} GB")
    print(f"VRAM total: {total:.1f} GB")
    if free < 8:
        print("⚠️  Less than 8GB free — DoRA may OOM. Try reducing batch size.")
    else:
        print("✅ Enough VRAM for DoRA.")

VRAM free : 15.5 GB
VRAM total: 15.6 GB
✅ Enough VRAM for DoRA.


In [11]:
# =========================
# FULL DoRA TRAINING CELL (cleaned + safer)
# =========================

import os
import gc
import pathlib
import torch
from dataclasses import dataclass
from typing import Any, Dict, List

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.generation.configuration_utils import GenerationConfig
from peft import LoraConfig, get_peft_model, TaskType

# ------------------------------------------------------------
# 0) Core variables
# ------------------------------------------------------------
MODEL_NAME = "openai/whisper-small"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Adjust these if not already defined earlier
OUTPUT_DIR_DORA = globals().get("OUTPUT_DIR_DORA", "/kaggle/working/whisper-dora-training")
ADAPTER_DIR_DORA = globals().get("ADAPTER_DIR_DORA", "/kaggle/working/whisper-dora-adapter")

# ------------------------------------------------------------
# 1) Force single GPU / non-distributed
# ------------------------------------------------------------
os.environ.pop("MASTER_ADDR", None)
os.environ.pop("MASTER_PORT", None)
os.environ.pop("WORLD_SIZE", None)
os.environ.pop("RANK", None)
os.environ.pop("LOCAL_RANK", None)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

cfg = pathlib.Path.home() / ".cache/huggingface/accelerate/default_config.yaml"
cfg.parent.mkdir(parents=True, exist_ok=True)
cfg.write_text("""\
compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
gpu_ids: '0'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
use_cpu: false
""")

gc.collect()
torch.cuda.empty_cache()

print(f"device: {device}")

# ------------------------------------------------------------
# 2) Load processor
# ------------------------------------------------------------
print(f"⬇️ Loading processor for {MODEL_NAME}...")
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
print("✅ Processor loaded")

# ------------------------------------------------------------
# 3) Collator
# ------------------------------------------------------------
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        # Keep float32; Trainer autocast/fp16 handles compute safely
        batch["input_features"] = batch["input_features"].float()

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        bos_id = self.processor.tokenizer.bos_token_id
        if labels.shape[1] > 0 and bos_id is not None:
            if (labels[:, 0] == bos_id).all():
                labels = labels[:, 1:]

        batch["labels"] = labels
        batch.pop("input_ids", None)
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# ------------------------------------------------------------
# 4) Check datasets exist
# ------------------------------------------------------------
if "train_hf" not in globals() or "val_hf" not in globals():
    raise NameError(
        "train_hf / val_hf are not defined. Re-run the feature extraction cell "
        "or reload them from disk before running this DoRA cell."
    )

# ------------------------------------------------------------
# 5) Clean datasets
# ------------------------------------------------------------
KEEP = {"input_features", "labels"}

def clean_ds(ds):
    drop_cols = [c for c in ds.column_names if c not in KEEP]
    if drop_cols:
        ds = ds.remove_columns(drop_cols)
    ds.reset_format()
    return ds

train_clean = clean_ds(train_hf)
val_clean = clean_ds(val_hf)

print("train columns:", train_clean.column_names)
print("val columns  :", val_clean.column_names)

# ------------------------------------------------------------
# 6) Load base Whisper model
# ------------------------------------------------------------
print(f"⬇️ Loading {MODEL_NAME} for DoRA...")
dora_base = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

if hasattr(dora_base.config, "suppress_tokens"):
    del dora_base.config.suppress_tokens
if hasattr(dora_base.config, "forced_decoder_ids"):
    del dora_base.config.forced_decoder_ids

dora_base.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
dora_base.generation_config.forced_decoder_ids = None
dora_base.generation_config.suppress_tokens = []

if hasattr(dora_base.config, "use_cache"):
    dora_base.config.use_cache = False

# Optional memory saver
dora_base.gradient_checkpointing_enable()

# ------------------------------------------------------------
# 7) Apply DoRA
# ------------------------------------------------------------
dora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    use_dora=True,
)

dora_model = get_peft_model(dora_base, dora_config)
dora_model.print_trainable_parameters()
dora_model.to(device)

# ------------------------------------------------------------
# 8) Sanity check
# ------------------------------------------------------------
print("\n🧪 Sanity forward pass...")
sb = {k: v.to(device) for k, v in data_collator([train_clean[0], train_clean[1]]).items()}

train_model_dora = dora_model.get_base_model()
train_model_dora.to(device)

with torch.no_grad():
    out = train_model_dora(**sb)

print(f"✅ Sanity OK — loss: {float(out.loss):.4f}")

del sb, out
gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 9) Training args (low-memory safer config)
# ------------------------------------------------------------
dora_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR_DORA,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    warmup_steps=50,
    num_train_epochs=3,
    eval_strategy="no",
    save_strategy="epoch",
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    predict_with_generate=False,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
)

# ------------------------------------------------------------
# 10) Trainer
# ------------------------------------------------------------
dora_trainer = Seq2SeqTrainer(
    model=train_model_dora,
    args=dora_args,
    train_dataset=train_clean,
    eval_dataset=val_clean,
    data_collator=data_collator,
)
# ------------------------------------------------------------
# 11) Train
# ------------------------------------------------------------
print("\n🚀 Training DoRA...")
train_result = dora_trainer.train()

# ------------------------------------------------------------
# 12) Save adapter
# ------------------------------------------------------------
os.makedirs(ADAPTER_DIR_DORA, exist_ok=True)
dora_model.save_pretrained(ADAPTER_DIR_DORA)
processor.save_pretrained(ADAPTER_DIR_DORA)

print(f"\n✅ DoRA adapter saved -> {ADAPTER_DIR_DORA}")

gc.collect()
torch.cuda.empty_cache()

device: cuda:0
⬇️ Loading processor for openai/whisper-small...
✅ Processor loaded
train columns: ['input_features', 'labels']
val columns  : ['input_features', 'labels']
⬇️ Loading openai/whisper-small for DoRA...
trainable params: 1,824,768 || all params: 243,559,680 || trainable%: 0.7492

🧪 Sanity forward pass...
✅ Sanity OK — loss: 4.8142

🚀 Training DoRA...


Step,Training Loss
20,5.731000
40,2.810500
60,1.613400
80,1.420300
100,1.126400
120,0.508300
140,0.290200
160,0.266100
180,0.258500
200,0.229900



✅ DoRA adapter saved -> /kaggle/working/whisper-dora-adapter


In [12]:
# ── Evaluate DoRA model ────────────────────────────────────────────────────────
from jiwer import wer, cer
from transformers.generation.configuration_utils import GenerationConfig

if hasattr(dora_model.config, 'suppress_tokens'):    del dora_model.config.suppress_tokens
if hasattr(dora_model.config, 'forced_decoder_ids'): del dora_model.config.forced_decoder_ids
dora_model.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
dora_model.generation_config.forced_decoder_ids = None
dora_model.generation_config.suppress_tokens    = []
dora_model.eval()

print(f'🔬 Evaluating DoRA on {N_EVAL} samples (same fixed subset as baseline)...')
dora_preds = [
    transcribe_whisper(dora_model, p)
    for p in tqdm(eval_subset['audio_path'].tolist(), desc='DoRA eval')
]

dora_wer = wer(refs, dora_preds)
dora_cer = cer(refs, dora_preds)
dora_improvement = (baseline_wer - dora_wer) / baseline_wer * 100
print(f'\n📊 MODEL B — Whisper-small + DoRA')
print(f'   WER : {dora_wer*100:.2f}%  (baseline was {baseline_wer*100:.2f}%)  → {dora_improvement:+.1f}% vs baseline')
print(f'   CER : {dora_cer*100:.2f}%')

🔬 Evaluating DoRA on 200 samples (same fixed subset as baseline)...


DoRA eval: 100%|██████████| 200/200 [01:49<00:00,  1.83it/s]


📊 MODEL B — Whisper-small + DoRA
   WER : 9.60%  (baseline was 16.27%)  → +41.0% vs baseline
   CER : 4.80%


In [ ]:
# ── Free DoRA model before loading Wav2Vec2 ───────────────────────────────────
import gc
del dora_model
torch.cuda.empty_cache()
gc.collect()
print('✅ DoRA model freed from GPU memory.')

In [ ]:
# Run this AFTER DoRA completes
import gc, torch
try: del dora_trainer
except: pass
try: del dora_model
except: pass
try: del dora_base
except: pass
try: del train_model_dora
except: pass
gc.collect()
torch.cuda.empty_cache()
free = torch.cuda.mem_get_info()[0] / 1e9
print(f"✅ DoRA cleared. VRAM free: {free:.1f} GB — ready for Wav2Vec2")

---
## 7️⃣ Model C — Wav2Vec2-base Full Fine-Tuning (~2 hrs)

**Architecture difference from Whisper:**

| | Whisper | Wav2Vec2 |
|---|---|---|
| Architecture | Encoder-Decoder | Encoder-only + CTC head |
| Pre-training data | 680K hours supervised | 960 hours LibriSpeech |
| Decoding | Autoregressive (token by token) | CTC (parallel) |
| Input | Log-mel spectrogram | Raw waveform |
| Fine-tuning style | LoRA on attention layers | Full fine-tune (CNN frozen) |

**What is CTC decoding?**
CTC (Connectionist Temporal Classification) predicts one character/token per audio
frame in parallel, then collapses repeated predictions and removes blank tokens.
It is faster than Whisper's autoregressive decoding but typically less accurate
on complex vocabulary.

**Why uppercase the text for Wav2Vec2?**
The `wav2vec2-base-960h` model was pre-trained to predict uppercase characters only.
Feeding it lowercase transcripts would cause training to diverge. We uppercase
the labels for training, then lowercase predictions for fair WER comparison.

In [13]:
import librosa
import torch
from datasets import Dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, TrainingArguments, Trainer
from dataclasses import dataclass
from typing import Dict, List

W2V_MODEL_NAME = 'facebook/wav2vec2-base-960h'
print(f'⬇️  Loading {W2V_MODEL_NAME}...')
w2v_processor = Wav2Vec2Processor.from_pretrained(W2V_MODEL_NAME)
w2v_model     = Wav2Vec2ForCTC.from_pretrained(W2V_MODEL_NAME)

w2v_model.freeze_feature_extractor()
w2v_model.to(device)

total     = sum(p.numel() for p in w2v_model.parameters())
trainable = sum(p.numel() for p in w2v_model.parameters() if p.requires_grad)
print(f'✅ Wav2Vec2 loaded: {total/1e6:.0f}M total, {trainable/1e6:.0f}M trainable ({trainable/total*100:.0f}%)')

def prepare_w2v(batch):
    audio, _ = librosa.load(batch['audio_path'], sr=16000)
    batch['input_values'] = w2v_processor(audio, sampling_rate=16000).input_values[0]
    batch['labels'] = w2v_processor.tokenizer(batch['text'].upper()).input_ids
    return batch

W2V_MAP = dict(
    remove_columns=['audio_path', 'text'],
    keep_in_memory=False,
    writer_batch_size=100,
    num_proc=1
)

print('🔄 Preparing Wav2Vec2 datasets (raw waveforms)...')
train_w2v = Dataset.from_pandas(train_df).map(prepare_w2v, **W2V_MAP)
val_w2v   = Dataset.from_pandas(val_df).map(prepare_w2v, **W2V_MAP)
print('✅ Wav2Vec2 datasets ready.')

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor

    def __call__(self, features: List[Dict]) -> Dict:
        inputs = self.processor.pad(
            [{'input_values': f['input_values']} for f in features],
            return_tensors='pt',
            padding=True
        )

        labels = self.processor.tokenizer.pad(
            [{'input_ids': f['labels']} for f in features],
            return_tensors='pt',
            padding=True
        )

        inputs['labels'] = labels['input_ids'].masked_fill(
            labels.attention_mask.ne(1), -100
        )
        return inputs

w2v_collator = DataCollatorCTCWithPadding(processor=w2v_processor)

⬇️  Loading facebook/wav2vec2-base-960h...


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

...

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Wav2Vec2 loaded: 94M total, 90M trainable (96%)
🔄 Preparing Wav2Vec2 datasets (raw waveforms)...


Map:   0%|          | 0/4884 [00:00<?, ? examples/s]

Map:   0%|          | 0/611 [00:00<?, ? examples/s]

✅ Wav2Vec2 datasets ready.


In [14]:
# ── Train Wav2Vec2 ────────────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer

# group_by_length=True: group samples of similar length into the same batch.
# This reduces padding waste (long and short sequences mixed = lots of PAD tokens).
# It's especially important for Wav2Vec2 whose input lengths vary widely.
w2v_args = TrainingArguments(
    output_dir=OUTPUT_DIR_W2V,
    group_by_length=True,           # reduces padding waste in variable-length audio
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,  # effective batch = 16
    learning_rate=1e-4,
    warmup_steps=100,
    num_train_epochs=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=True,
    logging_steps=20,
    report_to='none',
)

w2v_trainer = Trainer(
    model=w2v_model,
    args=w2v_args,
    train_dataset=train_w2v,
    eval_dataset=val_w2v,
    data_collator=w2v_collator,
    tokenizer=w2v_processor.feature_extractor,
)

print('🚀 Training Wav2Vec2 — ~2 hours on P100...')
w2v_trainer.train()

os.makedirs(W2V_SAVE_DIR, exist_ok=True)
w2v_model.save_pretrained(W2V_SAVE_DIR)
w2v_processor.save_pretrained(W2V_SAVE_DIR)
print(f'\n✅ Wav2Vec2 saved → {W2V_SAVE_DIR}')

🚀 Training Wav2Vec2 — ~2 hours on P100...


Epoch,Training Loss,Validation Loss
1,230.543800,220.137375
2,172.761500,160.706085
3,211.528800,156.439774
4,128.844900,135.273682
5,117.368100,135.554428
6,134.834800,119.116150
7,108.283900,112.829071
8,94.459200,112.724007
9,109.431000,118.117470
10,84.674100,117.524605


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr


✅ Wav2Vec2 saved → /kaggle/working/wav2vec2-saved


In [15]:
# ── Evaluate Wav2Vec2 ─────────────────────────────────────────────────────────
# CTC transcription is different from Whisper's generate():
#   1. Forward pass outputs logits: shape (1, time_steps, vocab_size)
#   2. argmax picks the highest-probability token at each time step
#   3. batch_decode applies CTC collapse: remove blanks, remove repeated tokens
#      e.g. [H,H,E,_,L,L,L,_,_,O] → 'HELLO'
# We lowercase the output to match our ground-truth refs (which are lowercase).
w2v_model.eval()

def transcribe_w2v(audio_path):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs   = w2v_processor(audio, sampling_rate=16000, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = w2v_model(**inputs).logits  # shape: (1, T, vocab_size)
    # CTC greedy decode: argmax over vocab at each frame, then collapse
    ids = torch.argmax(logits, dim=-1)
    return w2v_processor.batch_decode(ids)[0].strip().lower()

print(f'🔬 Evaluating Wav2Vec2 on {N_EVAL} samples (same fixed subset as all other models)...')
w2v_preds = [
    transcribe_w2v(p)
    for p in tqdm(eval_subset['audio_path'].tolist(), desc='Wav2Vec2 eval')
]

w2v_wer = wer(refs, w2v_preds)
w2v_cer = cer(refs, w2v_preds)
print(f'\n📊 MODEL C — Wav2Vec2-base (full fine-tune)')
print(f'   WER : {w2v_wer*100:.2f}%')
print(f'   CER : {w2v_cer*100:.2f}%')

🔬 Evaluating Wav2Vec2 on 200 samples (same fixed subset as all other models)...


Wav2Vec2 eval: 100%|██████████| 200/200 [00:07<00:00, 27.64it/s]


📊 MODEL C — Wav2Vec2-base (full fine-tune)
   WER : 16.85%
   CER : 10.25%


---
## 8️⃣ Final Results Table

In [37]:
# ── Cell 2: Paths ──────────────────────────────────────────────────────────────
LORA_ADAPTER_DIR = "/kaggle/input/datasets/aymendhieb1/whisper-adapter-final-v1/whisper_adapter_final"
DORA_ADAPTER_DIR = "/kaggle/input/datasets/aymendhieb1/whisper-dora-adapter-v1/whisper_dora_adapter"
W2V_DIR          = "/kaggle/input/datasets/aymendhieb1/wav2vec2-saved-v1/wav2vec2-saved"
MODEL_NAME       = "openai/whisper-small"
device           = "cuda"


In [24]:
# ── Debug 4: show exact file structure inside recordings ──────────────────────
import os

DATA_DIR = "/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent/recordings"

for split in os.listdir(DATA_DIR):
    split_dir = os.path.join(DATA_DIR, split)
    if not os.path.isdir(split_dir): continue
    files = os.listdir(split_dir)
    print(f"\n📁 {split}/ — {len(files)} files")
    print(f"   First 5: {files[:5]}")
    
    # check for subfolders
    subfolders = [f for f in files if os.path.isdir(os.path.join(split_dir, f))]
    if subfolders:
        print(f"   Subfolders: {subfolders}")
        for sf in subfolders[:2]:
            sf_path = os.path.join(split_dir, sf)
            sf_files = os.listdir(sf_path)
            print(f"   📁 {sf}/ — {sf_files[:5]}")


📁 validate/ — 385 files
   First 5: ['1249120_44294866_15891095.wav', '1249120_44263136_25998544.wav', '1249120_44263136_58938609.wav', '1249120_44294866_77416341.wav', '1249120_44323331_53313006.wav']

📁 test/ — 5895 files
   First 5: ['1249120_44101988_103474667.wav', '1249120_42210938_34058782.wav', '1249120_44093303_57046343.wav', '1249120_39740177_38724108.wav', '1249120_39740177_95271830.wav']

📁 train/ — 381 files
   First 5: ['1249120_44160489_107692984.wav', '1249120_44176037_39613511.wav', '1249120_44176037_85065458.wav', '1249120_44194084_23344116.wav', '1249120_44220382_48462181.wav']


In [25]:
# ── Debug 6: find the CSV ─────────────────────────────────────────────────────
import os

BASE = "/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent"

for root, dirs, files in os.walk(BASE):
    for f in files:
        if not f.endswith(".wav"):
            print(os.path.join(root, f))

/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent/overview-of-recordings.csv


In [26]:
# ── Cell 3: Load dataset + build eval subset ──────────────────────────────────
import pandas as pd, os, librosa
from sklearn.model_selection import train_test_split

BASE_DIR  = "/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent"
CSV_PATH  = os.path.join(BASE_DIR, "overview-of-recordings.csv")
AUDIO_DIR = os.path.join(BASE_DIR, "recordings")

# Load the CSV that maps filename → transcription
csv_df = pd.read_csv(CSV_PATH)
print("CSV columns:", csv_df.columns.tolist())
print(csv_df.head(3))

CSV columns: ['audio_clipping', 'audio_clipping:confidence', 'background_noise_audible', 'background_noise_audible:confidence', 'overall_quality_of_the_audio', 'quiet_speaker', 'quiet_speaker:confidence', 'speaker_id', 'file_download', 'file_name', 'phrase', 'prompt', 'writer_id']
   audio_clipping  audio_clipping:confidence background_noise_audible  \
0     no_clipping                     1.0000              light_noise   
1  light_clipping                     0.6803                 no_noise   
2     no_clipping                     1.0000                 no_noise   

   background_noise_audible:confidence  overall_quality_of_the_audio  \
0                               1.0000                          3.33   
1                               0.6803                          3.33   
2                               0.6655                          3.33   

     quiet_speaker  quiet_speaker:confidence  speaker_id  \
0  audible_speaker                       1.0    43453425   
1  audible_speak

In [27]:
# ── Cell 3: Load dataset + build eval subset ──────────────────────────────────
import pandas as pd, os, librosa
from sklearn.model_selection import train_test_split

BASE_DIR  = "/kaggle/input/datasets/paultimothymooney/medical-speech-transcription-and-intent/Medical Speech, Transcription, and Intent"
CSV_PATH  = os.path.join(BASE_DIR, "overview-of-recordings.csv")
AUDIO_DIR = os.path.join(BASE_DIR, "recordings")

# Load CSV
csv_df = pd.read_csv(CSV_PATH)

# Find the actual audio file by searching all split subfolders
def find_audio(filename):
    for split in ["train", "validate", "test"]:
        path = os.path.join(AUDIO_DIR, split, filename)
        if os.path.exists(path):
            return path
    return None

rows = []
for _, row in csv_df.iterrows():
    audio_path = find_audio(row["file_name"])
    if audio_path:
        rows.append({
            "audio_path": audio_path,
            "text": str(row["phrase"]).strip().lower()
        })

df = pd.DataFrame(rows)
print(f"Total rows found: {len(df)}")
assert len(df) > 0, "Still empty!"

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

eval_subset = test_df.sample(n=min(200, len(test_df)), random_state=42).reset_index(drop=True)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Eval subset: {len(eval_subset)} samples")
print(eval_subset[["audio_path","text"]].head(3))

Total rows found: 6661
Train: 5328 | Val: 666 | Test: 667
Eval subset: 200 samples
                                          audio_path  \
0  /kaggle/input/datasets/paultimothymooney/medic...   
1  /kaggle/input/datasets/paultimothymooney/medic...   
2  /kaggle/input/datasets/paultimothymooney/medic...   

                                                text  
0           i have a sharp pain in my lower stomach.  
1                             severe pain in the ear  
2  i feel something hurt me in taking breath and ...  


In [32]:
# ── Cell 4: WER/CER helper ────────────────────────────────────────────────────
from jiwer import wer, cer

def compute_metrics(refs, hyps):
    w = wer(refs, hyps)
    c = cer(refs, hyps)
    return w, c

In [33]:
# ── Cell 5: Baseline — Whisper-small, no fine-tuning ─────────────────────────
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor    = WhisperProcessor.from_pretrained(MODEL_NAME)
base_model   = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device).eval()

def transcribe_whisper(model, audio_path):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs   = processor(audio, sampling_rate=16000, return_tensors="pt").to(device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=128,
                             language="english", task="transcribe")
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip().lower()

refs, hyps = [], []
for _, row in eval_subset.iterrows():
    refs.append(row["text"])
    hyps.append(transcribe_whisper(base_model, row["audio_path"]))

baseline_wer, baseline_cer = compute_metrics(refs, hyps)
print(f"Baseline WER: {baseline_wer*100:.2f}%  CER: {baseline_cer*100:.2f}%")

del base_model; torch.cuda.empty_cache()

You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.


Baseline WER: 17.93%  CER: 7.78%


In [38]:
# ── Cell 6: Whisper + LoRA — load adapter and evaluate ────────────────────────
from peft import PeftModel

lora_base  = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
lora_model = PeftModel.from_pretrained(lora_base, LORA_ADAPTER_DIR).to(device).eval()

refs, hyps = [], []
for _, row in eval_subset.iterrows():
    refs.append(row["text"])
    hyps.append(transcribe_whisper(lora_model, row["audio_path"]))

lora_wer, lora_cer = compute_metrics(refs, hyps)
print(f"LoRA WER: {lora_wer*100:.2f}%  CER: {lora_cer*100:.2f}%")

del lora_base, lora_model; torch.cuda.empty_cache()

LoRA WER: 2.81%  CER: 1.64%


In [39]:
# ── Cell 7: Whisper + DoRA — same thing ───────────────────────────────────────
dora_base  = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
dora_model = PeftModel.from_pretrained(dora_base, DORA_ADAPTER_DIR).to(device).eval()

refs, hyps = [], []
for _, row in eval_subset.iterrows():
    refs.append(row["text"])
    hyps.append(transcribe_whisper(dora_model, row["audio_path"]))

dora_wer, dora_cer = compute_metrics(refs, hyps)
print(f"DoRA WER: {dora_wer*100:.2f}%  CER: {dora_cer*100:.2f}%")

del dora_base, dora_model; torch.cuda.empty_cache()

DoRA WER: 9.50%  CER: 4.52%


In [40]:
# ── Cell 8: Wav2Vec2 — load saved model and evaluate ─────────────────────────
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import numpy as np

w2v_processor = Wav2Vec2Processor.from_pretrained(W2V_DIR)
w2v_model     = Wav2Vec2ForCTC.from_pretrained(W2V_DIR).to(device).eval()

def transcribe_wav2vec(audio_path):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs   = w2v_processor(audio, sampling_rate=16000, return_tensors="pt",
                              padding=True).to(device)
    with torch.no_grad():
        logits = w2v_model(**inputs).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return w2v_processor.batch_decode(pred_ids)[0].strip().lower()

refs, hyps = [], []
for _, row in eval_subset.iterrows():
    refs.append(row["text"])
    hyps.append(transcribe_wav2vec(row["audio_path"]))

w2v_wer, w2v_cer = compute_metrics(refs, hyps)
print(f"Wav2Vec2 WER: {w2v_wer*100:.2f}%  CER: {w2v_cer*100:.2f}%")

del w2v_model; torch.cuda.empty_cache()

Wav2Vec2 WER: 14.61%  CER: 8.42%


In [41]:
# ── Cell 9: Final comparison table (your original code — now it will work!) ───
results = pd.DataFrame([
    {"Model": "Whisper-small", "Method": "No fine-tuning (baseline)",
     "Trainable params": "244M (100%)", "WER": f"{baseline_wer*100:.2f}%",
     "CER": f"{baseline_cer*100:.2f}%", "vs Baseline": "—"},
    {"Model": "Whisper-small", "Method": "LoRA (r=16, use_dora=False)",
     "Trainable params": "2.3M (~1%)", "WER": f"{lora_wer*100:.2f}%",
     "CER": f"{lora_cer*100:.2f}%",
     "vs Baseline": f"{(baseline_wer-lora_wer)/baseline_wer*100:+.1f}%"},
    {"Model": "Whisper-small", "Method": "DoRA (r=16, use_dora=True)",
     "Trainable params": "2.3M (~1%)", "WER": f"{dora_wer*100:.2f}%",
     "CER": f"{dora_cer*100:.2f}%",
     "vs Baseline": f"{(baseline_wer-dora_wer)/baseline_wer*100:+.1f}%"},
    {"Model": "Wav2Vec2-base", "Method": "Full fine-tuning (CNN frozen)",
     "Trainable params": "~90M", "WER": f"{w2v_wer*100:.2f}%",
     "CER": f"{w2v_cer*100:.2f}%", "vs Baseline": "N/A (different arch)"},
])

print("\n" + "="*75)
print("  🏆 FINAL RESULTS — Medical ASR Domain Adaptation Study")
print("="*75)
print(results.to_string(index=False))


  🏆 FINAL RESULTS — Medical ASR Domain Adaptation Study
        Model                        Method Trainable params    WER   CER          vs Baseline
Whisper-small     No fine-tuning (baseline)      244M (100%) 17.93% 7.78%                    —
Whisper-small   LoRA (r=16, use_dora=False)       2.3M (~1%)  2.81% 1.64%               +84.3%
Whisper-small    DoRA (r=16, use_dora=True)       2.3M (~1%)  9.50% 4.52%               +47.0%
Wav2Vec2-base Full fine-tuning (CNN frozen)             ~90M 14.61% 8.42% N/A (different arch)


In [42]:
# ── Save final results to CSV ─────────────────────────────────────────────────
import pandas as pd

results = pd.DataFrame([
    {"Model": "Whisper-small", "Method": "No fine-tuning (baseline)",
     "Trainable params": "244M (100%)", "WER": f"{baseline_wer*100:.2f}%",
     "CER": f"{baseline_cer*100:.2f}%", "vs Baseline": "—"},
    {"Model": "Whisper-small", "Method": "LoRA (r=16, use_dora=False)",
     "Trainable params": "2.3M (~1%)", "WER": f"{lora_wer*100:.2f}%",
     "CER": f"{lora_cer*100:.2f}%",
     "vs Baseline": f"{(baseline_wer - lora_wer)/baseline_wer*100:+.1f}%"},
    {"Model": "Whisper-small", "Method": "DoRA (r=16, use_dora=True)",
     "Trainable params": "2.3M (~1%)", "WER": f"{dora_wer*100:.2f}%",
     "CER": f"{dora_cer*100:.2f}%",
     "vs Baseline": f"{(baseline_wer - dora_wer)/baseline_wer*100:+.1f}%"},
    {"Model": "Wav2Vec2-base", "Method": "Full fine-tuning (CNN frozen)",
     "Trainable params": "~90M", "WER": f"{w2v_wer*100:.2f}%",
     "CER": f"{w2v_cer*100:.2f}%", "vs Baseline": "N/A (different arch)"},
])

results.to_csv("/kaggle/working/final_results.csv", index=False)
print("✅ Saved! Download from the Output panel on the right.")
print(results.to_string(index=False))

✅ Saved! Download from the Output panel on the right.
        Model                        Method Trainable params    WER   CER          vs Baseline
Whisper-small     No fine-tuning (baseline)      244M (100%) 17.93% 7.78%                    —
Whisper-small   LoRA (r=16, use_dora=False)       2.3M (~1%)  2.81% 1.64%               +84.3%
Whisper-small    DoRA (r=16, use_dora=True)       2.3M (~1%)  9.50% 4.52%               +47.0%
Wav2Vec2-base Full fine-tuning (CNN frozen)             ~90M 14.61% 8.42% N/A (different arch)


---
## 🎤 Inference Demo — Transcribe Any Audio File

In [44]:
# ── Inference: Load best model (LoRA wins with 2.81% WER) ────────────────────
import torch, librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers.generation.configuration_utils import GenerationConfig
from peft import PeftModel

scores = {'LoRA': lora_wer, 'DoRA': dora_wer}
best   = min(scores, key=scores.get)
BEST_ADAPTER_DIR = LORA_ADAPTER_DIR if best == 'LoRA' else DORA_ADAPTER_DIR
print(f'🏆 Best model: {best}  (WER = {scores[best]*100:.2f}%)')

# Load base + best adapter
inf_base      = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
inf_model     = PeftModel.from_pretrained(inf_base, BEST_ADAPTER_DIR).to(device).eval()
inf_processor = WhisperProcessor.from_pretrained(BEST_ADAPTER_DIR)
inf_model.generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
inf_model.generation_config.forced_decoder_ids = None
inf_model.generation_config.suppress_tokens    = []
print('✅ Ready for inference.')

def transcribe_file(audio_path):
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs   = inf_processor(audio, sampling_rate=16000, return_tensors='pt').to(device)
    with torch.no_grad():
        ids = inf_model.generate(
            **inputs,
            max_new_tokens=128,
            max_length=None,
            language='english',
            task='transcribe'
        )
    return inf_processor.batch_decode(ids, skip_special_tokens=True)[0].strip().lower()

# ── Demo on 5 test samples ────────────────────────────────────────────────────
print('\n🎤 Demo transcriptions from test set:')
print('-' * 65)
for _, row in test_df.head(5).iterrows():
    pred  = transcribe_file(row['audio_path'])
    match = '✅' if pred.strip() == row['text'].strip() else '❌'
    print(f'  {match} Truth : {row["text"]}')
    print(f'     Pred  : {pred}')
    print()

🏆 Best model: LoRA  (WER = 2.81%)
✅ Ready for inference.

🎤 Demo transcriptions from test set:
-----------------------------------------------------------------
  ✅ Truth : i feel like my heart is on fire.
     Pred  : i feel like my heart is on fire.

  ✅ Truth : i feel abdominal pain
     Pred  : i feel abdominal pain

  ✅ Truth : i think my wound is infected
     Pred  : i think my wound is infected

  ✅ Truth : i feel a tightness in my chest
     Pred  : i feel a tightness in my chest

  ✅ Truth : i do not feel better in my muscles
     Pred  : i do not feel better in my muscles



In [ ]:
# ── Transcribe your own audio file ────────────────────────────────────────────
# To use with your own recording:
#   1. Click the folder icon in the left panel of Kaggle
#   2. Upload your .wav file
#   3. Set MY_AUDIO to the uploaded path
#   4. Uncomment the two lines below and run

# MY_AUDIO = '/kaggle/working/my_recording.wav'   # ← change to your file path
# print(f'Transcription: {transcribe_file(MY_AUDIO)}')

print('ℹ️  Upload a .wav file and uncomment the lines above to transcribe it.')